### Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

### Task 1: Zomato ratings - detect outliers in 'user_rating' using the IQR method

In [ ]:
# Create a sample Zomato ratings dataset with a few outliers deliberately included
zomato_ratings = pd.DataFrame({
    "restaurant_id": range(1, 16),
    "user_rating": [4.2, 4.5, 3.8, 4.0, 4.6, 3.9, 4.1, 0.5, 4.3, 4.4,
                     3.7, 9.8, 4.0, 4.2, 3.9]
})

zomato_ratings.to_csv("zomato_ratings.csv", index=False)
zomato_df = pd.read_csv("zomato_ratings.csv")
print(zomato_df)

In [ ]:
# IQR method for outlier detection
Q1 = zomato_df["user_rating"].quantile(0.25)
Q3 = zomato_df["user_rating"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")

outliers = zomato_df[(zomato_df["user_rating"] < lower_bound) | (zomato_df["user_rating"] > upper_bound)]
print("\nDetected outlier indices:", outliers.index.tolist())
print("\nOutlier rows:")
print(outliers)

### Task 2: Boxplot for Swiggy 'order_amount' to visually identify outliers

In [ ]:
swiggy_orders = pd.DataFrame({
    "order_id": range(1, 21),
    "order_amount": [250, 300, 280, 320, 260, 310, 290, 270, 3200, 330,
                      340, 300, 275, 285, 295, 15, 305, 315, 265, 325]
})

plt.figure(figsize=(6, 5))
plt.boxplot(swiggy_orders["order_amount"], vert=True)
plt.title("Boxplot of Swiggy Order Amounts")
plt.ylabel("Order Amount (₹)")
plt.xticks([1], ["order_amount"])
plt.show()

# The points plotted beyond the whiskers (e.g. the very high ~3200 value and the
# very low ~15 value) are the visually identified outliers.

### Task 3: Winsorize Paytm 'transaction_amount' - cap above 95th and below 5th percentile

In [ ]:
paytm_df = pd.DataFrame({
    "transaction_id": range(1, 31),
    "transaction_amount": np.concatenate([
        np.random.uniform(100, 2000, 27),
        [50, 25000, 30000]  # a few extreme values
    ])
})

print("Statistics before winsorization:")
print(paytm_df["transaction_amount"].describe())

In [ ]:
lower_cap = paytm_df["transaction_amount"].quantile(0.05)
upper_cap = paytm_df["transaction_amount"].quantile(0.95)

print(f"5th percentile (lower cap): {lower_cap:.2f}")
print(f"95th percentile (upper cap): {upper_cap:.2f}")

paytm_df["transaction_amount_winsorized"] = paytm_df["transaction_amount"].clip(
    lower=lower_cap, upper=upper_cap
)

print("\nStatistics after winsorization:")
print(paytm_df["transaction_amount_winsorized"].describe())

### Task 4: Flipkart prices stored as strings with currency symbols - convert to numeric

In [ ]:
flipkart_prices = pd.DataFrame({
    "product": ["Phone", "Laptop", "Headphones", "Watch", "Tablet"],
    "price": ["₹1,299", "₹45,999", "₹899", "₹2,499.50", "₹12,000"]
})

print("Before conversion:")
print(flipkart_prices)
print(flipkart_prices["price"].dtype)

In [ ]:
# Remove all non-numeric characters (currency symbols, commas) except the decimal point,
# then convert to a numeric type
flipkart_prices["price"] = (
    flipkart_prices["price"]
    .str.replace(r"[^\d.]", "", regex=True)
    .astype(float)
)

print("After conversion:")
print(flipkart_prices)
print(flipkart_prices["price"].dtype)

### Task 5: Fix mixed-type 'is_premium' column in Spotify user DataFrame

In [ ]:
# The original messy data - a mix of boolean, string, and integer types
spotify_users = pd.DataFrame({
    "user_id": range(1, 8),
    "is_premium": [True, "True", 1, "yes", False, "no", 0]
})

print("Before fixing:")
print(spotify_users)
print(spotify_users["is_premium"].apply(type))

In [ ]:
def to_boolean(value):
    # Treat 'True', 1, and 'yes' (case-insensitive) as True; everything else as False
    if isinstance(value, str):
        return value.strip().lower() in ("true", "yes")
    return value == 1 or value is True

spotify_users["is_premium"] = spotify_users["is_premium"].apply(to_boolean).astype(bool)

print("After fixing:")
print(spotify_users)
print(spotify_users["is_premium"].dtype)